# Démonstration des requêtes SQL d'extraction (C9)

Exécute les 5 requêtes documentées dans
[`sql/extraction/`](../sql/extraction/) contre la base de staging
PostgreSQL, pour en visualiser les sorties directement — complément
interactif à
[`docs/architecture/requetes_sql_extraction.md`](../docs/architecture/requetes_sql_extraction.md),
qui documente l'objectif métier de chaque requête.

**Prérequis** : base de staging démarrée et peuplée
(`docker compose -f infra/docker/docker-compose.yml up -d`, puis
`./scripts/init_staging_db.sh` et `./scripts/load_historique.sh`).

Toutes les jointures entre tables FluxPro (`commandes`, `clients`,
`lignes_commande`, `stocks`, `entrepots`, `produits`, `expeditions`)
sont des `INNER JOIN` (`JOIN` en SQL standard) : chaque ligne du
résultat exige une correspondance des deux côtés (ex. une commande sans
client rattaché, ou un stock sans produit rattaché, ne doit pas exister
dans une base FluxPro cohérente — un `INNER JOIN` reflète cette
contrainte, plutôt qu'un `LEFT JOIN` qui masquerait silencieusement une
incohérence de données).


In [1]:
import sys

sys.path.insert(0, "..")  # pour importer datacore si le notebook est lance depuis notebooks/

from datacore.ingestion.fluxpro import connect

conn = connect()
conn.autocommit = True
print("Connecte a la base de staging.")


Connecte a la base de staging.


In [2]:
def run_query(sql_path, limit=10):
    """Execute un fichier .sql et affiche un apercu tabulaire du resultat.

    Args:
        sql_path: chemin du fichier .sql a executer.
        limit: nombre maximum de lignes affichees (le compte total
            reste, lui, exact).
    """
    with open(sql_path, encoding="utf-8") as f:
        sql = f.read()
    cur = conn.cursor()
    cur.execute(sql)
    columns = [d[0] for d in cur.description]
    rows = cur.fetchall()
    shown = rows[:limit]
    widths = [
        max(len(str(c)), max((len(str(r[i])) for r in shown), default=0))
        for i, c in enumerate(columns)
    ]
    print(f"{len(rows)} ligne(s) -- apercu des {len(shown)} premieres :\n")
    print(" | ".join(c.ljust(w) for c, w in zip(columns, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in shown:
        print(" | ".join(str(v).ljust(w) for v, w in zip(r, widths)))
    return rows


## 2.1 — Commandes par client et par période

Nombre de commandes et quantité totale par client et par mois (FluxPro : `commandes` INNER JOIN `clients` INNER JOIN `lignes_commande`).


In [3]:
run_query("../sql/extraction/01_commandes_par_client_periode.sql")

57 ligne(s) -- apercu des 10 premieres :

client      | mois                      | nb_commandes | quantite_totale
------------+---------------------------+--------------+----------------
FreshMarket | 2025-01-01 00:00:00+00:00 | 20           | 927            
FreshMarket | 2025-02-01 00:00:00+00:00 | 21           | 1190           
FreshMarket | 2025-03-01 00:00:00+00:00 | 27           | 1562           
FreshMarket | 2025-04-01 00:00:00+00:00 | 22           | 1281           
FreshMarket | 2025-05-01 00:00:00+00:00 | 40           | 2717           
FreshMarket | 2025-06-01 00:00:00+00:00 | 19           | 936            
FreshMarket | 2025-07-01 00:00:00+00:00 | 22           | 1549           
FreshMarket | 2025-08-01 00:00:00+00:00 | 19           | 1230           
FreshMarket | 2025-09-01 00:00:00+00:00 | 20           | 1154           
FreshMarket | 2025-10-01 00:00:00+00:00 | 26           | 1764           


## 2.2 — Stocks par entrepôt

Stocks actuels par entrepôt et produit, avec repère de rupture (`stocks` INNER JOIN `entrepots` INNER JOIN `produits`).


In [4]:
run_query("../sql/extraction/02_stocks_par_entrepot.sql")

90 ligne(s) -- apercu des 10 premieres :

entrepot             | sku       | libelle                  | quantite | date_maj   | en_rupture
---------------------+-----------+--------------------------+----------+------------+-----------
Entrepot Omega Lille | SKU-10001 | Plaquette de frein       | 16       | 2026-08-01 | False     
Entrepot Omega Lille | SKU-10002 | Disque de frein          | 708      | 2026-07-28 | False     
Entrepot Omega Lille | SKU-10003 | Filtre a huile           | 395      | 2026-07-29 | False     
Entrepot Omega Lille | SKU-10004 | Filtre a air             | 444      | 2026-07-29 | False     
Entrepot Omega Lille | SKU-10005 | Bougie d'allumage        | 117      | 2026-08-01 | False     
Entrepot Omega Lille | SKU-10006 | Amortisseur avant        | 30       | 2026-07-28 | False     
Entrepot Omega Lille | SKU-10007 | Courroie de distribution | 646      | 2026-07-28 | False     
Entrepot Omega Lille | SKU-10008 | Batterie 12V             | 687      | 2026-07-28 |

## 2.3 — Expéditions en retard

Expéditions livrées après leur date prévue (`expeditions` INNER JOIN `commandes` INNER JOIN `clients`).


In [5]:
run_query("../sql/extraction/03_expeditions_en_retard.sql")

88 ligne(s) -- apercu des 10 premieres :

tracking_number | transporteur          | client      | date_livraison_prevue | date_livraison_reelle | jours_retard
----------------+-----------------------+-------------+-----------------------+-----------------------+-------------
OMG0001333      | EcoRoute              | NordDrive   | 2025-08-30            | 2025-09-02            | 3           
OMG0000527      | RapidFret             | MedioTex    | 2025-07-30            | 2025-08-02            | 3           
OMG0000573      | TransUnion Logistique | FreshMarket | 2026-07-15            | 2026-07-18            | 3           
OMG0000588      | RapidFret             | MedioTex    | 2025-01-23            | 2025-01-26            | 3           
OMG0000600      | TransUnion Logistique | FreshMarket | 2026-04-07            | 2026-04-10            | 3           
OMG0000013      | EcoRoute              | MedioTex    | 2026-04-02            | 2026-04-05            | 3           
OMG0000131      | EcoR

## 2.4 — Délais et coûts par client (historique)

Délai moyen, coût moyen et taux de retard par client sur l'historique volumineux (`historique_expeditions`, sans jointure : source unique déjà consolidée).


In [6]:
run_query("../sql/extraction/04_historique_delais_couts_par_client.sql")

3 ligne(s) -- apercu des 3 premieres :

client      | nb_expeditions | delai_moyen_jours | cout_moyen_eur | taux_retard_pct
------------+----------------+-------------------+----------------+----------------
FreshMarket | 8297           | 2.2               | 48.26          | 11.2           
NordDrive   | 8291           | 2.2               | 48.46          | 11.0           
MedioTex    | 8412           | 2.2               | 48.43          | 10.6           


## 2.5 — Évolution annuelle du délai (historique)

Évolution annuelle du délai moyen de livraison par client.


In [7]:
run_query("../sql/extraction/05_evolution_delai_historique_par_annee.sql")

15 ligne(s) -- apercu des 10 premieres :

client      | annee | nb_expeditions | delai_moyen_jours
------------+-------+----------------+------------------
FreshMarket | 2022  | 1784           | 2.2              
FreshMarket | 2023  | 1799           | 2.2              
FreshMarket | 2024  | 1865           | 2.2              
FreshMarket | 2025  | 1823           | 2.2              
FreshMarket | 2026  | 1026           | 2.1              
MedioTex    | 2022  | 1801           | 2.2              
MedioTex    | 2023  | 1806           | 2.2              
MedioTex    | 2024  | 1831           | 2.2              
MedioTex    | 2025  | 1889           | 2.1              
MedioTex    | 2026  | 1085           | 2.2              


## Conclusion

Les résultats obtenus ici confirment ceux déjà documentés dans
`docs/architecture/requetes_sql_extraction.md`. Le point de vigilance
noté pour C11/C13 reste valable : `clients.nom` (FluxPro) et
`historique_expeditions.client` (libellé texte libre) devront être
rapprochés explicitement lors de la modélisation de la base de travail
consolidée et de l'entrepôt de données.


In [8]:
conn.close()